# Taps and Observers

Shows tap and span observer setup around a tiny capture for maintainer experimentation.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.types.SpecCompat",
    "tl.types.TargetManifestDiff",
    "tl.types.TensorLog",
    "tl.types.TensorSliceSpec",
    "tl.user_funcs",
    "tl.user_funcs.ActivationPostfunc",
    "tl.user_funcs.Any",
    "tl.user_funcs.BufferVisibilityLiteral",
    "tl.user_funcs.BundleStreamWriter",
    "tl.user_funcs.Callable",
    "tl.user_funcs.CaptureOptions",
    "tl.user_funcs.CodePanelOption",
    "tl.user_funcs.Dict",
    "tl.user_funcs.GradientPostfunc",
    "tl.user_funcs.InterventionReadyConflictError",
    "tl.user_funcs.List",
    "tl.user_funcs.MISSING",
    "tl.user_funcs.MissingType",
    "tl.user_funcs.ModelLog",
    "tl.user_funcs.Optional",
    "tl.user_funcs.OutputDeviceLiteral",
    "tl.user_funcs.Path",
    "tl.user_funcs.RunState",
    "tl.user_funcs.SaveOptions",
    "tl.user_funcs.StreamingOptions",
    "tl.user_funcs.TYPE_CHECKING",
    "tl.user_funcs.TorchLensIOError",
    "tl.user_funcs.TorchLensPostfuncError",
    "tl.user_funcs.TrainingModeConfigError",
    "tl.user_funcs.Tuple",
    "tl.user_funcs.Union",
    "tl.user_funcs.VisDirectionLiteral",
    "tl.user_funcs.VisInterventionModeLiteral",
    "tl.user_funcs.VisModeLiteral",
    "tl.user_funcs.VisNodeModeLiteral",
    "tl.user_funcs.VisNodePlacementLiteral",
    "tl.user_funcs.VisRendererLiteral",
    "tl.user_funcs.VisualizationOptions",
    "tl.user_funcs.capture_model_source_code",
    "tl.user_funcs.cast",
    "tl.user_funcs.check_model_and_input_variants",
    "tl.user_funcs.collections",
    "tl.user_funcs.decide_recording_of_batch",
    "tl.user_funcs.get_model_metadata",
    "tl.user_funcs.get_vars_of_type_from_obj",
    "tl.user_funcs.hashlib",
    "tl.user_funcs.json",
    "tl.user_funcs.list_logs",
    "tl.user_funcs.log_forward_pass",
    "tl.user_funcs.log_model_metadata",
    "tl.user_funcs.make_weak_model_ref",
    "tl.user_funcs.merge_capture_options",
    "tl.user_funcs.merge_save_options",
    "tl.user_funcs.merge_streaming_options",
    "tl.user_funcs.merge_visualization_options",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")